In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import tensorflow as tf
print('Version of tensorflow :',tf.__version__)

Version of tensorflow : 2.15.0


In [2]:
from keras.layers import Input,Dense,Flatten
from keras.models import Model
from keras.optimizers import Adam
from keras.applications.vgg16 import VGG16,preprocess_input
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
import numpy as np
import glob
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
from datetime import datetime
from keras.callbacks import ModelCheckpoint

In [3]:
IMAGE_SIZE = [ 224 , 224 , 3 ]

# Load the model
vgg = VGG16( include_top = False,
            input_shape = IMAGE_SIZE,
            weights = 'imagenet')

# Visualize the model
vgg.summary()

58889256/58889256 [==============================] - 2s 0us/step
Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                              

In [4]:
for  layer in vgg.layers:
    layer.trainable = False

In [5]:
# Flattened the last layer
x = Flatten()(vgg.output)

# Created a new layer as output
prediction = Dense( 7 , activation = 'softmax' )(x)

# Join it with the model
model = Model( inputs = vgg.input , outputs = prediction )

# Visualize the model again
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [6]:
adam=Adam()

# Compile the model
model.compile(loss='categorical_crossentropy',
              optimizer=adam,
              metrics=['accuracy'])


In [7]:
# For the train data generator
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# For the test data generator
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)


In [8]:
train_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Train'
test_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Test'

In [9]:
# train data
train_set = train_datagen.flow_from_directory(train_path,
                                            target_size = ( 224 , 224 ),
                                            batch_size = 32,
                                            class_mode = 'categorical')

# test data
test_set = test_datagen.flow_from_directory(test_path,
                                             target_size = ( 224 , 224 ),
                                            batch_size = 32,
                                            class_mode = 'categorical')

Found 6300 images belonging to 7 classes.
Found 1580 images belonging to 7 classes.


In [10]:
output_layer = model.layers[-1]  # Assuming the output layer is the last layer in the model
num_classes = output_layer.output_shape[-1]  # Number of units in the output layer

print("Number of classes in the output layer:", num_classes)


Number of classes in the output layer: 7


In [11]:
checkpoint = ModelCheckpoint(filepath = '/content/drive/MyDrive/Models/vgg16.h5' , verbose = 2 , save_best_only = True )
callbacks = [checkpoint]
start = datetime.now()
model_history = model.fit( train_set,validation_data = test_set,epochs = 5,steps_per_epoch = 197,validation_steps = 50,callbacks = callbacks)

duration = datetime.now() - start

print('Total elapsed time : ',duration)

Epoch 1/5
197/197 [==============================] - ETA: 0s - loss: 5.4571 - accuracy: 0.7371 
Epoch 1: val_loss improved from inf to 2.89460, saving model to /content/drive/MyDrive/Models/vgg16.h5


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


197/197 [==============================] - 4701s 24s/step - loss: 5.4571 - accuracy: 0.7371 - val_loss: 2.8946 - val_accuracy: 0.8380
Epoch 2/5
197/197 [==============================] - ETA: 0s - loss: 0.8461 - accuracy: 0.9400
Epoch 2: val_loss improved from 2.89460 to 2.20313, saving model to /content/drive/MyDrive/Models/vgg16.h5
197/197 [==============================] - 600s 3s/step - loss: 0.8461 - accuracy: 0.9400 - val_loss: 2.2031 - val_accuracy: 0.8728
Epoch 3/5
197/197 [==============================] - ETA: 0s - loss: 0.6034 - accuracy: 0.9573
Epoch 3: val_loss did not improve from 2.20313
197/197 [==============================] - 613s 3s/step - loss: 0.6034 - accuracy: 0.9573 - val_loss: 2.7929 - val_accuracy: 0.8772
Epoch 4/5
197/197 [==============================] - ETA: 0s - loss: 0.3843 - accuracy: 0.9668
Epoch 4: val_loss did not improve from 2.20313
197/197 [==============================] - 568s 3s/step - loss: 0.3843 - accuracy: 0.9668 - val_loss: 2.5367 - val_a